# Sentence Transformer

para este caso usaremos 

In [ ]:
from deep_translator import GoogleTranslator  # Para traducción automática
from langdetect import detect  # Para detectar el idioma de un texto
from sentence_transformers import SentenceTransformer  # Para generar embeddings con RoBERTa

import torch  # Para usar la GPU si está disponible
import os  # Para manejar rutas de archivos

import faiss  # Para búsquedas rápidas en el índice de embeddings
import numpy as np
import pandas as pd
import pickle
from pathlib import Path

PROJECT_ROOT = Path(f"{Path.cwd()}").resolve()

def translate_to_english(text):
    """
    Traduce la consulta del usuario a inglés si no está en inglés.
    """
    try:
        detected_lang = detect(text)
        if detected_lang != 'en':
            translated_text = GoogleTranslator(source='auto', target='en').translate(text)
            print(f"Traducido: '{text}' → '{translated_text}'")
            return translated_text
        return text
    except Exception as e:
        print(f"Error al detectar idioma: {e}")
        return text

file_path = r"..\data\processed\df_clean.parquet"
faiss_index_path = r"..\data\processed\games_embeddings_faiss_IP.index"
embeddings_file = r"..\data\processed\games_embeddings_sentrans.pkl"

df = pd.read_parquet(file_path)

print(df.columns)

df["info_compressed"] = ( df["name"].astype(str).fillna('') + " " + 
         df["release_date"].astype(str).fillna('') + " " +
         df["required_age"].astype(str).fillna('') + " " +
         df["price"].astype(str).fillna('') + " " +
         df["dlc_count"].astype(str).fillna('') + " " +
         df["about_the_game"].astype(str).fillna('') + " " +
         df["windows"].astype(str).fillna('') + " " +
         df["mac"].astype(str).fillna('') + " " +
         df["linux"].astype(str).fillna('') + " " +
         df["metacritic_score"].astype(str).fillna('') + " " +
         df["achievements"].astype(str).fillna('') + " " +
         df["recommendations"].astype(str).fillna('') + " " +
         df["supported_languages"].astype(str).fillna('') + " " +
         df["full_audio_languages"].astype(str).fillna('') + " " +
         df["developers"].astype(str).fillna('') + " " +
         df["categories"].astype(str).fillna('') + " " +
         df["genres"].astype(str).fillna('') + " " +
         df["estimated_owners"].astype(str).fillna('') + " " +
         df["average_playtime_forever"].astype(str).fillna('') + " " +
         df["peak_ccu"].astype(str).fillna('') + " " +
         df["tags"].astype(str).fillna('')
)

df["info_compressed"]

Index(['app_id', 'name', 'release_date', 'required_age', 'price', 'dlc_count',
       'detailed_description', 'about_the_game', 'short_description',
       'reviews', 'header_image', 'website', 'windows', 'mac', 'linux',
       'metacritic_score', 'achievements', 'recommendations',
       'supported_languages', 'full_audio_languages', 'developers',
       'publishers', 'categories', 'genres', 'screenshots', 'user_score',
       'positive', 'negative', 'estimated_owners', 'average_playtime_forever',
       'discount', 'peak_ccu', 'tags', 'text_description', 'has_text',
       'release_date_raw', 'release_year', 'tags_text'],
      dtype='str')


0         Black Dragon Mage Playtest 2023-08-01 0 0.0 0 ...
1         Supipara - Chapter 1 Spring Has Come! 2016-07-...
2         Mystery Solitaire The Black Raven 2019-05-06 0...
3         버튜버 파라노이아 - Vtuber Paranoia 2024-10-31 0 8.99 ...
4         Maze Quest VR 2025-04-24 0 4.99 0 Its not just...
                                ...                        
122606    完美传奇 2026-01-04 0 0.0 0 《完美传奇》游戏介绍 🔥【完美传奇——专属大...
122607    Poker Fate - ACG Texas Hold'em 2026-01-03 0 0....
122608    Adira Nusantara 2026-01-03 0 7.99 0 (Gif chara...
122609    A Lenda de Niterói 2026-01-04 0 2.09 0 Step in...
122610    [BEAT:KEEPER] 2026-01-05 0 0.0 0 Exhibited at ...
Name: info_compressed, Length: 122611, dtype: str

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando: {device.upper()}")

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)

print("Generando nuevos embeddings (vectores de 1024 dimensiones)...")
embeddings_list = model.encode(df["info_compressed"].tolist(), batch_size=256, show_progress_bar=True)

# 📌 Convertir embeddings a un array de NumPy en formato float32
embeddings_np = np.array(embeddings_list, dtype=np.float32)
d = embeddings_np.shape[1]  # Dimensión del vector (1024 en este caso)

# 📌 Crear el índice FAISS con `IndexFlatIP` (Producto Interno) sin normalización
index = faiss.IndexFlatIP(d)
index.add(embeddings_np)

# 📌 Guardar FAISS en archivo con manejo de errores
try:
    faiss.write_index(index, faiss_index_path)
    print(f"✅ FAISS `IndexFlatIP` guardado en {faiss_index_path}")
except Exception as e:
    print(f"❌ Error al guardar FAISS index: {e}")

# 📌 Guardar embeddings en Pickle para futuras consultas (🔥 SOLUCIONADO EL ERROR)
try:
    with open(embeddings_file, "wb") as f:
        pickle.dump(embeddings_list, f)  # Guardar correctamente como lista
    print(f"✅ Embeddings guardados en {embeddings_file}")
except Exception as e:
    print(f"❌ Error al guardar embeddings en Pickle: {e}")

Usando: CPU


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generando nuevos embeddings (vectores de 1024 dimensiones)...


KeyboardInterrupt: 

In [ ]:
query = "quiero un juegos de acción y terror"
query_translated = translate_to_english(query)

# 📌 Generar el embedding de la consulta
query_embedding = model.encode([query_translated], device=device, show_progress_bar=False)
query_embedding_np = np.array(query_embedding, dtype=np.float32)

# 📌 Realizar la búsqueda en FAISS (top 5 resultados)
k = 5
distances, indices = index.search(query_embedding_np, k)

# 📌 Mostrar resultados
if indices.size == 0:
    print("❌ No se encontraron resultados.")
else:
    results = []
    for i, idx in enumerate(indices[0]):
        sim_score = distances[0][i]
        song_info = df.iloc[idx].to_dict()
        song_info['similarity'] = sim_score
        results.append(song_info)

    results_df = pd.DataFrame(results)
    results_df
    #print(results_df[['song_name', 'artist_name', 'spotify_url', 'similarity']].to_string(index=False))

Traducido: 'quiero un juegos de acción y terror' → 'I want action and horror games'


NameError: name 'model' is not defined